# Stage 1 RF SHAP (TreeExplainer, wide noise classifier)

TreeExplainer SHAP for the canonical Stage 1 **RF** noise classifier on wide rows (animal × frequency).

- **Train:** Fit split only (`DataGroup == Train`)
- **Explain:** Validate + Test rows (held out from RF training)
- **Target:** positive class (`noise_cat == 1`)

Note: RF hyperparameters are tuned on Validate row accuracy; Validate SHAP reflects that tuning.

Outputs (3 figures × 2 cohorts, PNG + SVG):
- Beeswarm: feature types + frequency
- Bar: mean |SHAP| by feature type
- Beeswarm: feature types colored by true noise group

Parquet: `figures/cache/shap_stage1/`
Figures: `figures/shap_stage1/`


In [ ]:
from __future__ import annotations

from pathlib import Path

import shap

from utils.benchmark_metrics import apply_slide_rcparams
from utils.nn_stage2 import fit_stage1_wide_best
from utils.nn_stage2_data import (
    load_nn_stage2_data,
    splits_for_long_stage2,
    wide_stage1_fit,
    wide_stage1_val,
)
from utils.stage1_rf_shap import (
    COHORT_SLUGS_DECK,
    N_STAGE1_WIDE_FEATURES,
    STAGE1_SHAP_CACHE_DIR,
    STAGE1_SHAP_FIG_DIR,
    aggregated_feature_type_comparison_table_stage1,
    compute_cohort_stage1_rf_shap,
    export_stage1_shap_parquet,
    run_stage1_shap_deck_figures,
)
from utils.stage2_xgb_shap import full_feature_comparison_table

apply_slide_rcparams()
print("shap version:", shap.__version__)

CACHE_DIR = STAGE1_SHAP_CACHE_DIR
FIG_DIR = STAGE1_SHAP_FIG_DIR
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

data = load_nn_stage2_data()
splits = splits_for_long_stage2(data)

# Drift check: canonical selection should still be RF for both cohorts.
bb_fit = wide_stage1_fit(splits["bb_wide_train"])
bb_val = wide_stage1_val(splits["bb_wide_train"])
lib_fit = wide_stage1_fit(splits["lib_train"])
lib_val = wide_stage1_val(splits["lib_train"])

s1_bb_best = fit_stage1_wide_best(
    bb_fit, bb_val, splits["bb_wide_test"],
    data.noise_num_bb, data.noise_log_bb, verbose=False,
)
s1_lib_best = fit_stage1_wide_best(
    lib_fit, lib_val, splits["lib_test"],
    data.noise_num_lib, data.noise_log_lib, verbose=False,
)
assert s1_bb_best["stage1_model"] == s1_lib_best["stage1_model"] == "rf"
print("Stage 1 model selection: RF (both cohorts)")


In [ ]:
cohort_outputs: dict[str, dict] = {}
feature_table = None

for cohort_slug in COHORT_SLUGS_DECK:
    out = compute_cohort_stage1_rf_shap(cohort_slug, data, splits)
    cohort_outputs[cohort_slug] = {
        "shap": out["shap"],
        "feat": out["feat"],
        "meta": out["meta"],
    }
    feature_table = out["feature_table"]
    paths = export_stage1_shap_parquet(
        cohort_slug,
        out["shap"],
        out["meta"],
        feature_table=feature_table,
        cache_dir=CACHE_DIR,
    )
    meta = out["meta"]
    print(
        cohort_slug,
        "rows=", len(meta),
        "animals=", meta["animal_id"].nunique(),
        "shap cols=", out["shap"].shape[1],
        "parquet=", paths["shap"],
    )

assert feature_table is not None
assert out["shap"].shape[1] == N_STAGE1_WIDE_FEATURES


In [ ]:
print("Full features (sorted by pooled mean |SHAP|)")
display(full_feature_comparison_table(cohort_outputs, feature_table))

print("Aggregated feature types")
display(aggregated_feature_type_comparison_table_stage1(cohort_outputs, feature_table))


In [ ]:
paths = run_stage1_shap_deck_figures(
    cohort_outputs, FIG_DIR, feature_table=feature_table
)
print("wrote", len(paths), "figure bases to", FIG_DIR)
